In [ ]:
# ============================================================
# Практическое задание:
# Очистка текста, лемматизация, мешок слов и TF-IDF с Natasha
# ============================================================

import re
import math
from collections import Counter

import natasha as nt

# ------------------------------------------------------------
# 1. Инициализация минимально необходимых инструментов Natasha
# ------------------------------------------------------------

segmenter = nt.Segmenter()
morph_vocab = nt.MorphVocab()

emb = nt.NewsEmbedding()
morph_tagger = nt.NewsMorphTagger(emb)

# ------------------------------------------------------------
# 2. Загрузка текста
# ------------------------------------------------------------

with open("Текст.txt", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# 3. Передача текста в Natasha
# ------------------------------------------------------------

doc = nt.Doc(text)
doc.segment(segmenter)
doc.tag_morph(morph_tagger)

# ------------------------------------------------------------
# 4. Очистка и лемматизация
# ------------------------------------------------------------
# Оставляем только слова:
# - убираем пунктуацию
# - приводим к нижнему регистру
# - приводим к леммам
# - исключаем служебные символы и пустые токены

lemmas = []

for token in doc.tokens:
    token.lemmatize(morph_vocab)
    lemma = token.lemma.lower()

    # фильтрация: только буквы русского алфавита
    if re.fullmatch(r"[а-яё]+", lemma):
        lemmas.append(lemma)

# ------------------------------------------------------------
# 5. Мешок слов (Bag of Words)
# ------------------------------------------------------------
# Словарь: {слово: количество вхождений}

bow = Counter(lemmas)

print("Мешок слов (первые 20 слов):")
for word, count in bow.most_common(20):
    print(f"{word}: {count}")

# ------------------------------------------------------------
# 6. Расчёт TF (Term Frequency)
# ------------------------------------------------------------
# TF = частота слова / общее число слов

total_words = sum(bow.values())
tf = {word: count / total_words for word, count in bow.items()}

# ------------------------------------------------------------
# 7. Расчёт IDF (Inverse Document Frequency)
# ------------------------------------------------------------
# Так как у нас один документ:
# IDF = log(N / df) = log(1 / 1) = 0
# Для корректности используем сглаживание: log((N+1)/(df+1)) + 1

N = 1  # количество документов
idf = {word: math.log((N + 1) / (1 + 1)) + 1 for word in bow.keys()}

# ------------------------------------------------------------
# 8. Расчёт TF-IDF
# ------------------------------------------------------------
# TF-IDF = TF * IDF

tfidf = {word: tf[word] * idf[word] for word in bow.keys()}

print("\nTF-IDF (первые 20 слов):")
for word, value in sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"{word}: {value:.6f}")


Исходный текст:

Алисе наскучило сидеть с сестрой без дела на берегу реки; разок-другой она заглянула 
в книжку, которую читала сестра, но там не было ни картинок, ни разговоров. - Что толку в книжке, - подумала Алиса, - если в ней нет ни картинок, ни разговоров? 
Она сидела и размышляла, не встать ли ей и не нарвать ли цветов для венка; мысли ее 
текли медленно и несвязно - от жары ее клонило в сон. Конечно, сплести венок было бы очень 
приятно, но стоит ли ради этого подыматься? 
Вдруг мимо пробежал белый кролик с красными глазами. 
Конечно, ничего удивительного в этом не было. Правда, Кролик на бегу говорил: - Ах, боже мой, боже мой! Я опаздываю. 
Но и это не показалось Алисе особенно странным. (Вспоминая об этом позже, она 
подумала, что ей следовало бы удивиться, однако в тот миг все казалось ей вполне 
естественным.) Но, когда Кролик вдруг вынул часы из жилетного кармана и, взглянув на них, 
помчался дальше, Алиса вскочила на ноги. Ее тут осенило: ведь никогда раньше она не видел

[DocToken(stop=5, text='Алисе', id='1_1', head_id='1_2', rel='iobj', pos='PROPN', feats=<Anim,Dat,Fem,Sing>),
 DocToken(start=6, stop=15, text='наскучило', id='1_2', head_id='1_1', rel='nmod', pos='VERB', feats=<Perf,Ind,Plur,3,Fut,Fin,Act>),
 DocToken(start=16, stop=22, text='сидеть', id='1_3', head_id='1_1', rel='xcomp', pos='VERB', feats=<Imp,Inf,Act>),
 DocToken(start=23, stop=24, text='с', id='1_4', head_id='1_5', rel='case', pos='ADP'),
 DocToken(start=25, stop=32, text='сестрой', id='1_5', head_id='1_3', rel='obl', pos='NOUN', feats=<Anim,Ins,Fem,Sing>)]

Первые 5 предложений:


[DocSent(stop=161, text='Алисе наскучило сидеть с сестрой без дела на бере..., tokens=[...], spans=[...]),
 DocSent(start=162, stop=246, text='- Что толку в книжке, - подумала Алиса, - если в ..., tokens=[...], spans=[...]),
 DocSent(start=248, stop=386, text='Она сидела и размышляла, не встать ли ей и не нар..., tokens=[...]),
 DocSent(start=387, stop=468, text='Конечно, сплести венок было бы очень \nприятно, н..., tokens=[...]),
 DocSent(start=470, stop=522, text='Вдруг мимо пробежал белый кролик с красными глаза..., tokens=[...])]

Морфологический разбор первых 5 токенов:


[DocToken(stop=5, text='Алисе', id='1_1', head_id='1_2', rel='iobj', pos='PROPN', feats=<Anim,Dat,Fem,Sing>),
 DocToken(start=6, stop=15, text='наскучило', id='1_2', head_id='1_1', rel='nmod', pos='VERB', feats=<Perf,Ind,Plur,3,Fut,Fin,Act>),
 DocToken(start=16, stop=22, text='сидеть', id='1_3', head_id='1_1', rel='xcomp', pos='VERB', feats=<Imp,Inf,Act>),
 DocToken(start=23, stop=24, text='с', id='1_4', head_id='1_5', rel='case', pos='ADP'),
 DocToken(start=25, stop=32, text='сестрой', id='1_5', head_id='1_3', rel='obl', pos='NOUN', feats=<Anim,Ins,Fem,Sing>)]

Примеры нормализации словосочетаний:
Примеры лемматизации:
Найденные даты (в виде словарей):

Даты в удобном формате:

Словарь именованных сущностей (PER):

Список извлеченных имен:


['Алиса', 'Кролик', 'Нора', 'Дина', 'Белый Кролик']